In [1]:
import random
from typing import Dict, List


class PushdownAutomaton:

  def __init__(self, start_symbol: str, rules: Dict[str, List[List[str]]]):
    """Initializes a PDA from a Context-Free Grammar.

        Args:
            start_symbol: The initial variable Z (e.g., 'M').
            rules: Transposition transitions δ(q0, ε, variable) mapping each
              variable to a list of possible production rules (each rule is a
              list of tokens).
        """
    self.Z = start_symbol
    self.delta_transposition = rules
    self.variables = set(rules.keys())

  def generate_sentence(
      self, max_steps: int = 100, policy_fn=None, trace: bool = False
  ) -> str:
    """Implements Algorithm 1: PDA sentence generation for a given grammar G.

        Args:
            max_steps: Safeguard to prevent infinite recursion on infinite
              grammars.
            policy_fn: Optional function(variable, options, stack, sentence) ->
              chosen_rule. Defaults to uniform random selection if None.
            trace: If True, prints the step-by-step stack and sentence evolution.
        """
    # Initialisation (Algorithm 1)
    stack = [self.Z]
    w = []
    steps = 0

    if trace:
      print(f"Start:   Stack={stack} | Sentence=''\n" + "-" * 50)

    while stack:
      if steps >= max_steps:
        raise RuntimeError(
            f"Exceeded max_steps ({max_steps}). Derivation did not terminate."
        )

      # c <- pop(S)
      c = stack.pop()

      if c not in self.variables:
        # Transcription transition: c ∈ Σ (Terminal symbol)
        w.append(c)
        if trace:
          print(
              f"Transcrip (pop '{c}'): Stack={stack} | Sentence='{''.join(w)}'"
          )
      else:
        # Transposition transition: c ∈ V (Variable)
        options = self.delta_transposition[c]
        if policy_fn is not None:
          chosen_rule = policy_fn(c, options, stack, w)
        else:
          chosen_rule = random.choice(options)

        # Push symbols in reverse order so the leftmost symbol is on top
        for token in reversed(chosen_rule):
          stack.append(token)

        if trace:
          rule_str = "".join(chosen_rule)
          print(
              f"Transpos  ({c} -> {rule_str}): Stack={stack} |"
              f" Sentence='{''.join(w)}'"
          )

      steps += 1

    # Stack is empty -> PDA transitions to accepting state q1
    return "".join(w)


# =====================================================================
# Example: Instantiating PDA D3 from Grammar G3 (Section 3.1 & 3.2)
# Rule: M -> (M ⊙ M) | (MM) | A | I | J
# =====================================================================

G3_rules = {
    "M": [
        ["(", "M", "⊙", "M", ")"],
        ["(", "M", "M", ")"],
        ["A"],
        ["I"],
        ["J"],
    ]
}

pda_D3 = PushdownAutomaton(start_symbol="M", rules=G3_rules)

# 1. Generate a random valid matrix formula from L(G3)
print("Randomly generated formula:")
formula = pda_D3.generate_sentence(max_steps=50, trace=True)
print(f"\nFinal Accepted Sentence: {formula}\n")

# 2. Reproduce the exact derivation of (J⊙(AA)) from Figure 1 of the paper
deterministic_choices = [
    ["(", "M", "⊙", "M", ")"],  # 1st M -> (M⊙M)
    ["J"],  # 2nd M -> J
    ["(", "M", "M", ")"],  # 3rd M -> (MM)
    ["A"],  # 4th M -> A
    ["A"],  # 5th M -> A
]
choice_iter = iter(deterministic_choices)

print("=" * 50)
print("Reproducing Figure 1: (J⊙(AA))")
fig1_formula = pda_D3.generate_sentence(
    policy_fn=lambda var, opts, s, w: next(choice_iter), trace=True
)
print(f"\nFinal Accepted Sentence: {fig1_formula}")

Randomly generated formula:
Start:   Stack=['M'] | Sentence=''
--------------------------------------------------
Transpos  (M -> I): Stack=['I'] | Sentence=''
Transcrip (pop 'I'): Stack=[] | Sentence='I'

Final Accepted Sentence: I

Reproducing Figure 1: (J⊙(AA))
Start:   Stack=['M'] | Sentence=''
--------------------------------------------------
Transpos  (M -> (M⊙M)): Stack=[')', 'M', '⊙', 'M', '('] | Sentence=''
Transcrip (pop '('): Stack=[')', 'M', '⊙', 'M'] | Sentence='('
Transpos  (M -> J): Stack=[')', 'M', '⊙', 'J'] | Sentence='('
Transcrip (pop 'J'): Stack=[')', 'M', '⊙'] | Sentence='(J'
Transcrip (pop '⊙'): Stack=[')', 'M'] | Sentence='(J⊙'
Transpos  (M -> (MM)): Stack=[')', ')', 'M', 'M', '('] | Sentence='(J⊙'
Transcrip (pop '('): Stack=[')', ')', 'M', 'M'] | Sentence='(J⊙('
Transpos  (M -> A): Stack=[')', ')', 'M', 'A'] | Sentence='(J⊙('
Transcrip (pop 'A'): Stack=[')', ')', 'M'] | Sentence='(J⊙(A'
Transpos  (M -> A): Stack=[')', ')', 'A'] | Sentence='(J⊙(A'
Transcrip (pop